In [1]:
import os
import subprocess

command = "source ~/.zprofile && env"
proc = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    shell=True,
    executable="/bin/zsh"
)

for line in proc.stdout:
    key, _, value = line.decode().partition("=")
    os.environ[key] = value.strip()

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-everything",
        ]
    }
})

# Later in your async code
tools = await client.get_tools()

In [3]:
tools

[StructuredTool(name='echo', description='Echoes back the input string', args_schema={'type': 'object', 'properties': {'message': {'type': 'string', 'description': 'Message to echo'}}, 'required': ['message'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x141089440>),
 StructuredTool(name='get-annotated-message', description='Demonstrates how annotations can be used to provide metadata about content.', args_schema={'type': 'object', 'properties': {'messageType': {'type': 'string', 'enum': ['error', 'success', 'debug'], 'description': 'Type of message to demonstrate different annotation patterns'}, 'includeImage': {'type': 'boolean', 'default': False, 'description': 'Whether to include an example image'}}, 'required': ['messageType'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, response_fo

In [4]:
from langchain.agents import create_agent
agent = create_agent("openai:gpt-4.1", tools)
result = await agent.ainvoke({"messages": [{"role": "user", "content": "Give me all environement variables"}]})
print(f"Result: = {result}")

Result: = {'messages': [HumanMessage(content='Give me all environement variables', additional_kwargs={}, response_metadata={}, id='844b8167-03d8-4eef-a749-c92e35e09880'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 630, 'total_tokens': 640, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_1a2c4a5ede', 'id': 'chatcmpl-CzyEVpOPNLs1uaDL7yuVVC0DuuCRT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bd9cb-ee65-7122-8885-7f4d334bba43-0', tool_calls=[{'name': 'get-env', 'args': {}, 'id': 'call_at0Ymvjbb52mIE8Hw7KrwsLG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 630, 'output_tokens

In [6]:
print("stream mode test:")
async for event in agent.astream({
    "messages": [{"role": "user", "content": "Give me all environement variables"}]
}):
    print(event)

stream mode test:
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 1165, 'total_tokens': 1177, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_1a2c4a5ede', 'id': 'chatcmpl-Czy1svYWOL7ol4GTa4yn8tz95q0PB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bd9bf-f9e5-7793-aa5a-13fb2b02c8ff-0', tool_calls=[{'name': 'list_allowed_directories', 'args': {}, 'id': 'call_GEmB2Qwlmo1fOfCtkNyOOVHP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1165, 'output_tokens': 12, 'total_tokens': 1177, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'

In [5]:
print("stream mode test:")
async for event in agent.astream({
    "messages": [{"role": "user", "content": "Give me all environement variables"}]
}):
    # Check if event has the structure you showed
    if isinstance(event, dict) and 'model' in event:
        model_data = event['model']
        if 'messages' in model_data and isinstance(model_data['messages'], list):
            for msg in model_data['messages']:
                # Check if this is an AIMessage-like object
                if hasattr(msg, 'content') and msg.content:
                    print(msg.content)
                # Alternative: if it's a dictionary representation
                elif isinstance(msg, dict) and 'content' in msg and msg['content']:
                    print(msg['content'])

stream mode test:
Here are all the environment variables currently set:

- NODE: /opt/homebrew/Cellar/node@20/20.19.4/bin/node
- INIT_CWD: /Users/welcome/Library/Mobile Documents/com~apple~CloudDocs/Tech_Learn/Tech_Repos/python_envs/A2A/A2A_event_agents/A2A_Examples/mcp_expe/server-everything-mcp
- TERM: xterm-color
- SHELL: /bin/zsh
- npm_config_global_prefix: /opt/homebrew
- COLOR: 0
- npm_config_noproxy: 
- npm_config_local_prefix: /Users/welcome/Library/Mobile Documents/com~apple~CloudDocs/Tech_Learn/Tech_Repos/python_envs/A2A/A2A_event_agents/A2A_Examples/mcp_expe/server-everything-mcp
- USER: welcome
- npm_config_globalconfig: /opt/homebrew/etc/npmrc
- __CF_USER_TEXT_ENCODING: 0x1F5:0x0:0x0
- npm_execpath: /opt/homebrew/Cellar/node@20/20.19.4/lib/node_modules/npm/bin/npm-cli.js
- PATH: (long PATH with many directories; let me know if you want to see the entire value)
- npm_package_json: /Users/welcome/Library/Mobile Documents/com~apple~CloudDocs/Tech_Learn/Tech_Repos/python_envs/